In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Python Programs/StudentsPerformance.csv")
df

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75
...,...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,88,99,95
996,male,group C,high school,free/reduced,none,62,55,55
997,female,group C,high school,free/reduced,completed,59,71,65
998,female,group D,some college,standard,completed,68,78,77


In [ ]:
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [ ]:
df.tail( )

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
995,female,group E,master's degree,standard,completed,88,99,95
996,male,group C,high school,free/reduced,none,62,55,55
997,female,group C,high school,free/reduced,completed,59,71,65
998,female,group D,some college,standard,completed,68,78,77
999,female,group D,some college,free/reduced,none,77,86,86


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race/ethnicity               1000 non-null   object
 2   parental level of education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test preparation course      1000 non-null   object
 5   math score                   1000 non-null   int64 
 6   reading score                1000 non-null   int64 
 7   writing score                1000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 62.6+ KB


In [ ]:
df.describe()

,math score,reading score,writing score
count,1000.00000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000
std,15.16308,14.600192,15.195657
min,0.00000,17.000000,10.000000
25%,57.00000,59.000000,57.750000
50%,66.00000,70.000000,69.000000
75%,77.00000,79.000000,79.000000
max,100.00000,100.000000,100.000000


In [ ]:
df.shape

(1000, 8)

In [ ]:
df.isnull().sum()

,0
gender,0
race/ethnicity,0
parental level of education,0
lunch,0
test preparation course,0
math score,0
reading score,0
writing score,0


In [ ]:
score_columns = ['math score', 'reading score', 'writing score']
score_columns

['math score', 'reading score', 'writing score']

In [ ]:
df[score_columns].quantile(0.25)

,0.25
math score,57.00
reading score,59.00
writing score,57.75


In [ ]:
df[score_columns].quantile(0.50)

,0.5
math score,66.0
reading score,70.0
writing score,69.0


In [ ]:
df[score_columns].quantile(0.75)

,0.75
math score,77.0
reading score,79.0
writing score,79.0


In [ ]:
df.columns

Index(['gender', 'race/ethnicity', 'parental level of education', 'lunch',
       'test preparation course', 'math score', 'reading score',
       'writing score'],
      dtype='object')

In [ ]:
df['Total Score'] = df['math score'] + df['reading score'] + df['writing score']
df['Percentage'] = (df['Total Score']/300)*100

## 1. Clean categorical features
Gender, Race/Ethnicity, Parental Level of Education, Lunch, Test Prep Course

In [ ]:
categorical_columns = ['gender', 'race/ethnicity', 'parental level of education',
                       'lunch', 'test preparation course']

# Inspect the values before cleaning
for col in categorical_columns:
    print(f"{col}: {sorted(df[col].unique())}")

In [ ]:
# Standardise text: remove stray spaces, collapse repeated spaces, lower-case
for col in categorical_columns:
    df[col] = (df[col].astype(str)
                      .str.strip()
                      .str.replace(r'\s+', ' ', regex=True)
                      .str.lower())

# Fix known spelling variants (no-ops if the data is already clean)
df['gender'] = df['gender'].replace({'m': 'male', 'f': 'female'})
df['lunch'] = df['lunch'].replace({'free/reduced': 'free/reduced', 'free / reduced': 'free/reduced'})
df['test preparation course'] = df['test preparation course'].replace({'complete': 'completed'})

# Nicer display labels
df['gender'] = df['gender'].str.title()
df['race/ethnicity'] = df['race/ethnicity'].str.title()                      # 'Group B'
df['parental level of education'] = df['parental level of education'].str.title()
df['lunch'] = df['lunch'].str.title()
df['test preparation course'] = df['test preparation course'].str.title()

# Missing values / duplicates
print("Missing values:\n", df[categorical_columns].isnull().sum(), "\n")
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
# Verify the cleaned categories
for col in categorical_columns:
    print(df[col].value_counts(), "\n")

# Store as category dtype (saves memory, makes group-bys cleaner)
df[categorical_columns] = df[categorical_columns].astype('category')
df.dtypes

## 2. Statistical measures for subject scores
Mean, median, standard deviation and quartiles

In [ ]:
stats = df[score_columns].agg(['mean', 'median', 'std'])
quartiles = df[score_columns].quantile([0.25, 0.50, 0.75])
quartiles.index = ['Q1 (25%)', 'Q2 (50%)', 'Q3 (75%)']

summary_stats = pd.concat([stats, quartiles]).round(2)
summary_stats.loc['IQR (Q3-Q1)'] = summary_stats.loc['Q3 (75%)'] - summary_stats.loc['Q1 (25%)']
summary_stats

## 3. Total marks and percentage
Already computed above (`Total Score`, `Percentage`). Re-run here so the cell is safe if the data was cleaned/reloaded, and add a performance band.

In [ ]:
df['Total Score'] = df[score_columns].sum(axis=1)
df['Percentage'] = (df['Total Score'] / (100 * len(score_columns))).mul(100).round(2)

# Optional performance band based on percentage
df['Performance Band'] = pd.cut(df['Percentage'],
                                bins=[-0.01, 40, 60, 75, 90, 100],
                                labels=['Fail (<40)', 'Average (40-60)', 'Good (60-75)',
                                        'Very Good (75-90)', 'Excellent (90+)'])

print(df[['Total Score', 'Percentage']].describe().round(2))
df[score_columns + ['Total Score', 'Percentage', 'Performance Band']].head()

## 4. Detect extreme performance outliers
Two methods per subject (and for total score):
* **IQR rule** - outside `Q1 - 1.5*IQR` / `Q3 + 1.5*IQR` (*mild*), or `1.5` replaced by `3` (*extreme*)
* **Z-score** - absolute z-score above 3

In [ ]:
analysis_columns = score_columns + ['Total Score']

def outlier_bounds(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

rows = []
for col in analysis_columns:
    lo, hi = outlier_bounds(df[col], 1.5)
    elo, ehi = outlier_bounds(df[col], 3.0)
    z = (df[col] - df[col].mean()) / df[col].std()
    rows.append({
        'subject': col,
        'lower bound': round(lo, 2),
        'upper bound': round(hi, 2),
        'IQR outliers': int(((df[col] < lo) | (df[col] > hi)).sum()),
        'extreme (3xIQR)': int(((df[col] < elo) | (df[col] > ehi)).sum()),
        'z-score outliers (|z|>3)': int((z.abs() > 3).sum()),
    })
outlier_summary = pd.DataFrame(rows).set_index('subject')
outlier_summary

In [ ]:
# Flag each student per subject (IQR rule) and combine
for col in score_columns:
    lo, hi = outlier_bounds(df[col], 1.5)
    df[f'{col} outlier'] = np.where(df[col] < lo, 'Low',
                             np.where(df[col] > hi, 'High', 'Normal'))

lo, hi = outlier_bounds(df['Total Score'], 1.5)
df['Total Score outlier'] = np.where(df['Total Score'] < lo, 'Low',
                              np.where(df['Total Score'] > hi, 'High', 'Normal'))

flag_cols = [f'{c} outlier' for c in score_columns]
df['Outlier Subjects'] = (df[flag_cols] != 'Normal').sum(axis=1)

# Students who are outliers in at least one subject
outliers = df[(df['Outlier Subjects'] > 0) | (df['Total Score outlier'] != 'Normal')]
print("Students flagged as outliers:", len(outliers))
outliers.sort_values('Total Score').head(15)

In [ ]:
# Students who are outliers in 2 or more subjects (consistently extreme performance)
multi = df[df['Outlier Subjects'] >= 2]
print("Outliers in 2+ subjects:", len(multi))
multi[score_columns + ['Total Score', 'Percentage'] + flag_cols]

In [ ]:
# Visualise: box plots make the IQR outliers visible
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df[score_columns], ax=axes[0])
axes[0].set_title('Subject scores - box plot')
axes[0].set_ylabel('Score')

sns.boxplot(y=df['Percentage'], ax=axes[1], color='lightgreen')
axes[1].set_title('Overall percentage - box plot')
plt.tight_layout()
plt.show()

## Summary
* Categorical columns are standardised (trimmed, consistently cased, duplicates removed) and stored as `category`.
* Mean, median, standard deviation and quartiles are collected in `summary_stats`.
* `Total Score` (out of 300) and `Percentage` are derived features, with a `Performance Band`.
* Outliers are detected with the IQR rule and z-scores per subject and for total score; `outliers` holds the flagged students.